In [2]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

con = duckdb.connect("../dbt_project/fitness.duckdb", read_only=True)

def q(sql):
    """Run SQL against the dbt database and return a pandas DataFrame."""
    return con.sql(sql).df()

STRAVA, GARMIN = "#FC4C02", "#007CC3"   # brand colors for charts

In [ ]:
share = q("""
    select week_start, brand, share_of_search
    from fct_weekly_share_of_search
    where language = 'English'
      and week_start >= '2020-01-01'
      and week_start < date_trunc('week', current_date)   -- drop the incomplete current week
""")
share["week_start"] = pd.to_datetime(share["week_start"])

weekly = share.pivot(index="week_start", columns="brand", values="share_of_search")
smooth = weekly.rolling(13).mean()

fig, ax = plt.subplots(figsize=(11, 5))
for brand in smooth.columns:                       # all competitors in grey
    ax.plot(smooth.index, smooth[brand], lw=1, label=brand)
# ax.plot(smooth.index, smooth["Strava"], color=STRAVA, lw=2.5, label="Strava")
# ax.plot(smooth.index, smooth["Garmin"], color=GARMIN, lw=2.5, label="Garmin")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title("Share of search, fitness brands (English Wikipedia, 13-week average)")
ax.legend()
plt.show()